### Importando os Pacotes Utilitários
***

In [2]:
import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow
import os

### Verificando as Configurações do Ambiente de Desenvolvimento
***

In [3]:
# verificando o diretório de desenvolvimento
HOME = os.getcwd()
print(HOME)

/content


In [4]:
# verificando as configurações do CUDA para treinamento do modelo em GPU
!nvidia-smi

Tue Sep 12 19:32:15 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   31C    P0    46W / 400W |      0MiB / 40960MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [5]:
# verificando a compatibilidade do framework com o ambiente de desenvolvimento (CPU/GPU)
ultralytics.checks()

Ultralytics YOLOv8.0.176 🚀 Python-3.10.12 torch-2.0.1+cu118 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 26.4/78.2 GB disk)


### Baixando o Conjunto de Imagens Pré-processado do RoboFLow
***

In [ ]:
# baixando o conjunto de imagens do roboflow, utilizando a API integrada
API_KEY = os.environ.get('ROBOFLOW_API_KEY')
if not API_KEY:
    raise RuntimeError('Defina a variável de ambiente ROBOFLOW_API_KEY antes de baixar o conjunto.')
WORKSPACE = 'alysson-machado'
PROJECT = 'towers-instance-segmentation-v2'
FORMAT = 'yolov8'
VERSION = 4
rf = Roboflow(api_key = API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download(FORMAT)

### Preparando o Modelo para Treinamento

In [7]:
def define_version(directory, model_type = 'x'):
    '''
    Retorna o tipo de modelo da YOLOv8 de segmentação pré-treinado.

    Args:
        directory (str) --> diretório com os modelos salvos
        model_type (str) --> tipo do modelo (n, s, m, l, x)
    Return
        model_value (str) --> nome do modelo pré-treinado para treinamento
    '''

    return directory + 'yolov8' + model_type + '-seg.pt'

In [12]:
pretrained_model_dir = 'modelos-pretreinados-segmentacao/'
# instanciando o modelo treinável da YOLOv8
model = YOLO(define_version(pretrained_model_dir, 'x'))
print(model)

In [9]:
# === CONFIGURAÇÃO DO AMBIENTE DE TREINAMENTO ===

# salvar o modelo treinado ao decorrer das épocas
SAVE = True
# configuração do ambiente de treinamento (CUDA = 0, CPU = 1)
DEVICE = 0
# definindo os "worker threads" da GPU
WORKERS = 8
# utilizar o modelo pré-treinando carregado
# (o carregamento foi feito previamente)
PRETRAINED = False
# semente de geração do treinamento dos pesos para reprodutibilidade
SEED = 42
# validar o modelo após cada época de treinamento
VAL = True
# aplicar aumento de dados pelo pŕopio Dataloader
# (o aumento de dados foi aplicado pelo RoboFlow)
AUGMENT = False

### Treinamento do Modelo Instanciado
***

In [10]:
# === COMPILAÇÃO DO MODELO INSTANCIADO ===

# métrica de otimização do decrescimento da função de custo
# [SGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp, auto]
OPTIMIZER = 'SGD'
# taxas de aprendizado inicial e final (lr0*lr1) (i.e. SGD=1E-2, Adam=1E-3)
LR0, LRF = 0.01, 0.01
# taxa de momento do gradiente estocástico
MOMENTUM = 0.937
# taxa de dropout para inativação dos neurônios de saída
DROPOUT = 0

In [11]:
# === HIPER-PARÂMETROS DO MODELO ===

# arquivo de acesso ao conjunto de imagens
DATA = 'datasets/Towers-Instance-Segmentation-V2-4/data.yaml'
# número de épocas de treinamento
EPOCHS = 150
# épocas de observação para ativação do "early stoppoing"
PATIENCE = 50
# quantidade de lotes de imagens por etapa de treinamento
BATCH = -1
# dimensão das imagens de entrada
IMGSZ = 960

In [13]:
# treinando o modelo instanciado (configurações personalizadas)
model.train(data = DATA, epochs = EPOCHS, patience = PATIENCE, batch = BATCH,
            imgsz = IMGSZ, save = SAVE,device = DEVICE, workers = WORKERS,
            pretrained = PRETRAINED, optimizer = OPTIMIZER, seed = SEED,
            lr0 = LR0, lrf = LRF, momentum = MOMENTUM, dropout = DROPOUT,
            val = VAL, augment = AUGMENT)

Ultralytics YOLOv8.0.176 🚀 Python-3.10.12 torch-2.0.1+cu118 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
engine/trainer: task=segment, mode=train, model=modelos-pretreinados-segmentacao/yolov8x-seg.pt, data=/content/datasets/Towers-Instance-Segmentation-V2-4/data.yaml, epochs=150, patience=50, batch=-1, imgsz=960, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=None, exist_ok=False, pretrained=False, optimizer=SGD, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, overlap_mask=True, mask_ratio=4, dropout=0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, stream_buffer=False, line_width=None, visualize=False, augment=False, agnostic_nms=False, c

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ec6aee87df0>
fitness: 1.3706176909110144
keys: ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(M)', 'metrics/recall(M)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)']
maps: array([     1.3252])
names: {0: 'towers'}
plot: True
results_dict: {'metrics/precision(B)': 0.8669166488175682, 'metrics/recall(B)': 0.8407079646017699, 'metrics/mAP50(B)': 0.8758719641222299, 'metrics/mAP50-95(B)': 0.6790665184359728, 'metrics/precision(M)': 0.8852806032233066, 'metrics/recall(M)': 0.8584070796460177, 'metrics/mAP50(M)': 0.9035916790647545, 'metrics/mAP50-95(M)': 0.646123844444378, 'fitness': 1.3706176909110144}
save_dir: PosixPath('runs/segment/train')
seg: ultralytics.utils.metrics.Metric object
speed: {'preprocess': 1.4637005329132